### GeoRSCLIP

In [4]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', "3")

import io
import os
import time
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import open_clip

# ==========================================
# Configuration (Adjust paths as needed)
# ==========================================
CKPT_PATH = "/data/susanket/vlm/sota/ckpt/RS5M_ViT-B-32.pt"  # Predefined checkpoint path
BENCHMARK_ROOT = "/data/susanket/vlm/eval"
MODEL_ARCH = "ViT-B-32"                  # Update architecture if using ViT-L-14, etc.
BATCH_SIZE = 64
IMAGE_SIZE = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ==========================================
# 1. Load GeoRSCLIP Model & Tokenizer
# ==========================================
print(f"Initializing OpenCLIP architecture: {MODEL_ARCH}")
model, _, image_transform = open_clip.create_model_and_transforms(MODEL_ARCH, pretrained=None)

print(f"Loading checkpoint weights from: {CKPT_PATH}")
checkpoint = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

if "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
elif "model" in checkpoint:
    state_dict = checkpoint["model"]
else:
    state_dict = checkpoint

# Clean up DDP prefix if present
state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

model.load_state_dict(state_dict, strict=False)
model = model.to(device)
model.eval()

tokenizer = open_clip.get_tokenizer(MODEL_ARCH)
print("Model and tokenizer successfully loaded!")


# ==========================================
# 2. Datasets
# ==========================================
class ImageFolderFlat(Dataset):
    def __init__(self, root: str, transform, image_size: int = 224):
        self.root = Path(root)
        self.transform = transform
        self.image_size = image_size
        self.samples: List[Tuple[Path, str]] = []
        self.class_names: List[str] = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        self.class_to_idx = {name: i for i, name in enumerate(self.class_names)}

        for cls_name in self.class_names:
            cls_dir = self.root / cls_name
            for img_path in sorted(cls_dir.iterdir()):
                if img_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}:
                    self.samples.append((img_path, cls_name))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, cls_name = self.samples[idx]
        try:
            image = Image.open(path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (self.image_size, self.image_size))
        image = self.transform(image)
        return image, self.class_to_idx[cls_name]


class ParquetCaptionDataset(Dataset):
    def __init__(self, parquet_path: str, transform, image_size: int = 224):
        self.df = pd.read_parquet(parquet_path)
        self.transform = transform
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        caption = str(row["caption"])
        img_bytes = row["image"]

        try:
            image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        except Exception:
            image = Image.new("RGB", (self.image_size, self.image_size))

        image = self.transform(image)
        return image, caption


# ==========================================
# 3. Feature Extraction & Metrics Functions
# ==========================================
def encode_images_in_batches(model, dataloader: DataLoader, device: torch.device):
    model.eval()
    all_embeddings, all_targets = [], []
    with torch.no_grad():
        for images, targets in dataloader:
            images = images.to(device)
            embeddings = model.encode_image(images)
            embeddings = F.normalize(embeddings, dim=-1)
            all_embeddings.append(embeddings.cpu().numpy())
            
            if isinstance(targets, torch.Tensor):
                all_targets.append(targets.numpy())
            else:
                all_targets.extend(targets)

    if len(all_targets) > 0 and isinstance(all_targets[0], np.ndarray):
        all_targets = np.concatenate(all_targets, axis=0)

    return np.concatenate(all_embeddings, axis=0), all_targets


def encode_prompts(model, tokenizer, prompts: List[str], device: torch.device, batch_size: int = 128):
    model.eval()
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(prompts), batch_size):
            batch_prompts = prompts[i:i + batch_size]
            tokens = tokenizer(batch_prompts).to(device)
            embeddings = model.encode_text(tokens)
            embeddings = F.normalize(embeddings, dim=-1)
            all_embeddings.append(embeddings.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0)


def compute_retrieval_metrics(sim_matrix: np.ndarray, is_1to1: bool = False, img_labels: np.ndarray = None):
    n_text, n_img = sim_matrix.shape

    t2i_r1, t2i_r5, t2i_r10 = 0.0, 0.0, 0.0
    for i in range(n_text):
        ranks = np.argsort(-sim_matrix[i, :])
        matched = (ranks == i) if is_1to1 else (img_labels[ranks] == i)
        hit_ranks = np.where(matched)[0]
        if len(hit_ranks) > 0:
            if hit_ranks[0] < 1: t2i_r1 += 1.0
            if hit_ranks[0] < 5: t2i_r5 += 1.0
            if hit_ranks[0] < 10: t2i_r10 += 1.0

    t2i_res = {"T2I_R@1": t2i_r1 / n_text, "T2I_R@5": t2i_r5 / n_text, "T2I_R@10": t2i_r10 / n_text}

    i2t_r1, i2t_r5, i2t_r10 = 0.0, 0.0, 0.0
    for j in range(n_img):
        ranks = np.argsort(-sim_matrix[:, j])
        matched = (ranks == j) if is_1to1 else (ranks == img_labels[j])
        hit_ranks = np.where(matched)[0]
        if len(hit_ranks) > 0:
            if hit_ranks[0] < 1: i2t_r1 += 1.0
            if hit_ranks[0] < 5: i2t_r5 += 1.0
            if hit_ranks[0] < 10: i2t_r10 += 1.0

    i2t_res = {"I2T_R@1": i2t_r1 / n_img, "I2T_R@5": i2t_r5 / n_img, "I2T_R@10": i2t_r10 / n_img}
    return {**t2i_res, **i2t_res}


def compute_classification_accuracy(sim_matrix: np.ndarray, img_labels: np.ndarray):
    logits = sim_matrix.T  
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    top1_correct, top5_correct, n_samples = 0, 0, len(img_labels)

    for i in range(n_samples):
        ranking = np.argsort(-probs[i])
        if img_labels[i] == ranking[0]:
            top1_correct += 1
        if img_labels[i] in ranking[:5]:
            top5_correct += 1

    return {
        "Acc@1": top1_correct / n_samples,
        "Acc@5": top5_correct / min(5, probs.shape[1]) if probs.shape[1] >= 5 else top1_correct / n_samples
    }


# ==========================================
# 4. Run Evaluation Suite
# ==========================================
benchmark_root = Path(BENCHMARK_ROOT)
class_dir = benchmark_root / "image-class"
cap_dir = benchmark_root / "image-cap"

class_results = {}
cap_results = {}

if class_dir.exists():
    for ds in sorted([d for d in class_dir.iterdir() if d.is_dir()]):
        print(f"\n[Image-Class] Evaluating {ds.name}...", end=" ", flush=True)
        t0 = time.time()
        dataset = ImageFolderFlat(str(ds), transform=image_transform, image_size=IMAGE_SIZE)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
        
        img_emb, img_labels = encode_images_in_batches(model, loader, device)
        prompts = [f"{cls.replace('_', ' ')}" for cls in dataset.class_names]
        txt_emb = encode_prompts(model, tokenizer, prompts, device)
        
        sim_matrix = txt_emb @ img_emb.T
        retrieval_metrics = compute_retrieval_metrics(sim_matrix, is_1to1=False, img_labels=img_labels)
        cls_metrics = compute_classification_accuracy(sim_matrix, img_labels)
        
        class_results[ds.name] = {"num_images": len(dataset), "num_classes": len(dataset.class_names), **retrieval_metrics, **cls_metrics}
        print(f"done ({time.time()-t0:.1f}s) | Acc@1: {class_results[ds.name]['Acc@1']*100:.2f}%")

if cap_dir.exists():
    for p_file in sorted(list(cap_dir.glob("*.parquet"))):
        ds_name = p_file.stem
        print(f"\n[Image-Cap] Evaluating {ds_name}...", end=" ", flush=True)
        t0 = time.time()
        dataset = ParquetCaptionDataset(str(p_file), transform=image_transform, image_size=IMAGE_SIZE)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
        
        img_emb, captions = encode_images_in_batches(model, loader, device)
        txt_emb = encode_prompts(model, tokenizer, captions, device)
        
        sim_matrix = txt_emb @ img_emb.T
        retrieval_metrics = compute_retrieval_metrics(sim_matrix, is_1to1=True)
        
        cap_results[ds_name] = {"num_images": len(dataset), "T2I_R@1": retrieval_metrics["T2I_R@1"], "I2T_R@1": retrieval_metrics["I2T_R@1"]}
        print(f"done ({time.time()-t0:.1f}s) | T2I R@1: {cap_results[ds_name]['T2I_R@1']*100:.2f}%")


# ==========================================
# 5. Print Output Tables
# ==========================================
if class_results:
    print("\n" + "=" * 115)
    print("IMAGE-CLASS BENCHMARKS EVALUATION RESULTS")
    print("=" * 115)
    print(f"{'Dataset':<20} | {'Images':<8} | {'Classes':<8} | {'Acc@1':<8} | {'Acc@5':<8} | {'T2I R@1':<8} | {'I2T R@1':<8}")
    print("-" * 115)
    class_avg = defaultdict(float)
    for ds_name, r in sorted(class_results.items()):
        print(f"{ds_name:<20} | {r['num_images']:<8,} | {r['num_classes']:<8} | {r['Acc@1']*100:6.2f}%  | {r['Acc@5']*100:6.2f}%  | {r['T2I_R@1']*100:6.2f}%  | {r['I2T_R@1']*100:6.2f}%")
        for k in ["Acc@1", "Acc@5", "T2I_R@1", "I2T_R@1"]: class_avg[k] += r[k]
    print("-" * 115)
    for k in class_avg: class_avg[k] /= len(class_results)
    print(f"{'AVERAGE':<20} | {'-':<8} | {'-':<8} | {class_avg['Acc@1']*100:6.2f}%  | {class_avg['Acc@5']*100:6.2f}%  | {class_avg['T2I_R@1']*100:6.2f}%  | {class_avg['I2T_R@1']*100:6.2f}%")
    print("=" * 115)

if cap_results:
    print("\n" + "=" * 65)
    print("IMAGE-CAPTION BENCHMARKS EVALUATION RESULTS")
    print("=" * 65)
    print(f"{'Dataset':<25} | {'Images':<8} | {'T2I R@1':<8} | {'I2T R@1':<8}")
    print("-" * 65)
    cap_avg = defaultdict(float)
    for ds_name, r in sorted(cap_results.items()):
        print(f"{ds_name:<25} | {r['num_images']:<8,} | {r['T2I_R@1']*100:6.2f}%  | {r['I2T_R@1']*100:6.2f}%")
        for k in ["T2I_R@1", "I2T_R@1"]: cap_avg[k] += r[k]
    print("-" * 65)
    for k in cap_avg: cap_avg[k] /= len(cap_results)
    print(f"{'AVERAGE':<25} | {'-':<8} | {cap_avg['T2I_R@1']*100:6.2f}%  | {cap_avg['I2T_R@1']*100:6.2f}%")
    print("=" * 65)

Using device: cuda
Initializing OpenCLIP architecture: ViT-B-32


Loading checkpoint weights from: /data/susanket/vlm/sota/ckpt/RS5M_ViT-B-32.pt
Model and tokenizer successfully loaded!

[Image-Class] Evaluating CLRS... done (11.9s) | Acc@1: 65.76%

[Image-Class] Evaluating NWPU_images... done (4.4s) | Acc@1: 85.62%

[Image-Class] Evaluating Optimal31... done (2.3s) | Acc@1: 78.66%

[Image-Class] Evaluating RSC11... done (2.1s) | Acc@1: 65.30%

[Image-Class] Evaluating RSICB128... done (36.2s) | Acc@1: 32.39%

[Image-Class] Evaluating RSSCN7... done (3.6s) | Acc@1: 79.46%

[Image-Class] Evaluating WHU-RS19... done (2.2s) | Acc@1: 87.46%

[Image-Cap] Evaluating RSICD... done (26.8s) | T2I R@1: 2.10%

[Image-Cap] Evaluating RSITMD... done (12.2s) | T2I R@1: 2.70%

[Image-Cap] Evaluating UCM... done (6.3s) | T2I R@1: 1.81%

IMAGE-CLASS BENCHMARKS EVALUATION RESULTS
Dataset              | Images   | Classes  | Acc@1    | Acc@5    | T2I R@1  | I2T R@1 
--------------------------------------------------------------------------------------------------------

### GitRSCLIP

In [ ]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', "3")

import io
import time
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import AutoProcessor, AutoModel

# ==========================================
# Configuration
# ==========================================
MODEL_NAME = "lcybuaa/Git-RSCLIP"
BENCHMARK_ROOT = "/data/susanket/vlm/eval"
BATCH_SIZE = 64
IMAGE_SIZE = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ==========================================
# 1. Load Git-RSCLIP Model & Processor
# ==========================================
print(f"Loading Git-RSCLIP model and processor: {MODEL_NAME}")
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

processor = AutoProcessor.from_pretrained(MODEL_NAME)
print("Model and processor successfully loaded!")


# ==========================================
# 2. Datasets
# ==========================================
class ImageFolderFlatDataset(Dataset):
    """Loads images from class subfolders (raw PIL Images for processor handling)."""
    def __init__(self, root: str, image_size: int = 224):
        self.root = Path(root)
        self.image_size = image_size
        self.samples: List[Tuple[Path, str]] = []
        self.class_names: List[str] = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        self.class_to_idx = {name: i for i, name in enumerate(self.class_names)}

        for cls_name in self.class_names:
            cls_dir = self.root / cls_name
            for img_path in sorted(cls_dir.iterdir()):
                if img_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}:
                    self.samples.append((img_path, cls_name))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, cls_name = self.samples[idx]
        try:
            image = Image.open(path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (self.image_size, self.image_size))
        return image, self.class_to_idx[cls_name]


class ParquetCaptionDataset(Dataset):
    """Loads image-caption pairs from a Parquet file (raw PIL Images for processor handling)."""
    def __init__(self, parquet_path: str, image_size: int = 224):
        self.df = pd.read_parquet(parquet_path)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        caption = str(row["caption"])
        img_bytes = row["image"]

        try:
            image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        except Exception:
            image = Image.new("RGB", (self.image_size, self.image_size))

        return image, caption


# ==========================================
# 3. Feature Extraction & Metrics Functions
# ==========================================
def encode_images_in_batches(model, processor, dataloader: DataLoader, device: torch.device):
    model.eval()
    all_embeddings, all_targets = [], []
    
    with torch.no_grad():
        for images, targets in dataloader:
            pil_images = list(images) if not isinstance(images, list) else images
            
            inputs = processor(images=pil_images, return_tensors="pt", padding=True).to(device)
            outputs = model.get_image_features(**inputs)
            embeddings = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs
            
            embeddings = F.normalize(embeddings, dim=-1)
            all_embeddings.append(embeddings.cpu().numpy())
            
            # Flatten and collect targets properly regardless of type
            if isinstance(targets, torch.Tensor):
                all_targets.extend(targets.cpu().numpy().tolist())
            elif isinstance(targets, (list, np.ndarray)):
                all_targets.extend(list(targets))
            else:
                all_targets.append(int(targets))

    img_embeddings = np.concatenate(all_embeddings, axis=0)
    img_labels = np.array(all_targets, dtype=int)

    return img_embeddings, img_labels


def encode_prompts(model, processor, prompts: List[str], device: torch.device, batch_size: int = 128):
    model.eval()
    all_embeddings = []
    
    with torch.no_grad():
        for i in range(0, len(prompts), batch_size):
            batch_prompts = prompts[i:i + batch_size]
            inputs = processor(text=batch_prompts, return_tensors="pt", padding=True).to(device)
            outputs = model.get_text_features(**inputs)
            
            # Extract tensor from Hugging Face model output object if necessary
            embeddings = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs
            
            embeddings = F.normalize(embeddings, dim=-1)
            all_embeddings.append(embeddings.cpu().numpy())
            
    return np.concatenate(all_embeddings, axis=0)


def compute_retrieval_metrics(sim_matrix: np.ndarray, is_1to1: bool = False, img_labels: np.ndarray = None):
    n_text, n_img = sim_matrix.shape

    t2i_r1, t2i_r5, t2i_r10 = 0.0, 0.0, 0.0
    for i in range(n_text):
        ranks = np.argsort(-sim_matrix[i, :])
        matched = (ranks == i) if is_1to1 else (img_labels[ranks] == i)
        hit_ranks = np.where(matched)[0]
        if len(hit_ranks) > 0:
            if hit_ranks[0] < 1: t2i_r1 += 1.0
            if hit_ranks[0] < 5: t2i_r5 += 1.0
            if hit_ranks[0] < 10: t2i_r10 += 1.0

    t2i_res = {"T2I_R@1": t2i_r1 / n_text, "T2I_R@5": t2i_r5 / n_text, "T2I_R@10": t2i_r10 / n_text}

    i2t_r1, i2t_r5, i2t_r10 = 0.0, 0.0, 0.0
    for j in range(n_img):
        ranks = np.argsort(-sim_matrix[:, j])
        matched = (ranks == j) if is_1to1 else (ranks == img_labels[j])
        hit_ranks = np.where(matched)[0]
        if len(hit_ranks) > 0:
            if hit_ranks[0] < 1: i2t_r1 += 1.0
            if hit_ranks[0] < 5: i2t_r5 += 1.0
            if hit_ranks[0] < 10: i2t_r10 += 1.0

    i2t_res = {"I2T_R@1": i2t_r1 / n_img, "I2T_R@5": i2t_r5 / n_img, "I2T_R@10": i2t_r10 / n_img}
    return {**t2i_res, **i2t_res}


def compute_classification_accuracy(sim_matrix: np.ndarray, img_labels: np.ndarray):
    logits = sim_matrix.T  
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    top1_correct, top5_correct, n_samples = 0, 0, len(img_labels)

    for i in range(n_samples):
        ranking = np.argsort(-probs[i])
        if img_labels[i] == ranking[0]:
            top1_correct += 1
        if img_labels[i] in ranking[:5]:
            top5_correct += 1

    return {
        "Acc@1": top1_correct / n_samples,
        "Acc@5": top5_correct / min(5, probs.shape[1]) if probs.shape[1] >= 5 else top1_correct / n_samples
    }


# ==========================================
# 4. Run Evaluation Suite
# ==========================================
benchmark_root = Path(BENCHMARK_ROOT)
class_dir = benchmark_root / "image-class"
cap_dir = benchmark_root / "image-cap"

class_results = {}
cap_results = {}

def custom_collate(batch):
    # Custom collate to prevent PyTorch from trying to stack raw PIL images automatically
    images = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    if isinstance(targets[0], str):
        return images, targets
    return images, np.array(targets)

if class_dir.exists():
    for ds in sorted([d for d in class_dir.iterdir() if d.is_dir()]):
        print(f"\n[Image-Class] Evaluating {ds.name}...", end=" ", flush=True)
        t0 = time.time()
        dataset = ImageFolderFlatDataset(str(ds), image_size=IMAGE_SIZE)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=custom_collate)
        
        img_emb, img_labels = encode_images_in_batches(model, processor, loader, device)
        prompts = [f"{cls.replace('_', ' ')}" for cls in dataset.class_names]
        txt_emb = encode_prompts(model, processor, prompts, device)
        
        sim_matrix = txt_emb @ img_emb.T
        retrieval_metrics = compute_retrieval_metrics(sim_matrix, is_1to1=False, img_labels=img_labels)
        cls_metrics = compute_classification_accuracy(sim_matrix, img_labels)
        
        class_results[ds.name] = {"num_images": len(dataset), "num_classes": len(dataset.class_names), **retrieval_metrics, **cls_metrics}
        print(f"done ({time.time()-t0:.1f}s) | Acc@1: {class_results[ds.name]['Acc@1']*100:.2f}%")

if cap_dir.exists():
    for p_file in sorted(list(cap_dir.glob("*.parquet"))):
        ds_name = p_file.stem
        print(f"\n[Image-Cap] Evaluating {ds_name}...", end=" ", flush=True)
        t0 = time.time()
        dataset = ParquetCaptionDataset(str(p_file), image_size=IMAGE_SIZE)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=custom_collate)
        
        img_emb, captions = encode_images_in_batches(model, processor, loader, device)
        txt_emb = encode_prompts(model, processor, captions, device)
        
        sim_matrix = txt_emb @ img_emb.T
        retrieval_metrics = compute_retrieval_metrics(sim_matrix, is_1to1=True)
        
        cap_results[ds_name] = {"num_images": len(dataset), "T2I_R@1": retrieval_metrics["T2I_R@1"], "I2T_R@1": retrieval_metrics["I2T_R@1"]}
        print(f"done ({time.time()-t0:.1f}s) | T2I R@1: {cap_results[ds_name]['T2I_R@1']*100:.2f}%")


# ==========================================
# 5. Print Output Tables
# ==========================================
if class_results:
    print("\n" + "=" * 115)
    print("IMAGE-CLASS BENCHMARKS EVALUATION RESULTS (Git-RSCLIP)")
    print("=" * 115)
    print(f"{'Dataset':<20} | {'Images':<8} | {'Classes':<8} | {'Acc@1':<8} | {'Acc@5':<8} | {'T2I R@1':<8} | {'I2T R@1':<8}")
    print("-" * 115)
    class_avg = defaultdict(float)
    for ds_name, r in sorted(class_results.items()):
        print(f"{ds_name:<20} | {r['num_images']:<8,} | {r['num_classes']:<8} | {r['Acc@1']*100:6.2f}%  | {r['Acc@5']*100:6.2f}%  | {r['T2I_R@1']*100:6.2f}%  | {r['I2T_R@1']*100:6.2f}%")
        for k in ["Acc@1", "Acc@5", "T2I_R@1", "I2T_R@1"]: class_avg[k] += r[k]
    print("-" * 115)
    for k in class_avg: class_avg[k] /= len(class_results)
    print(f"{'AVERAGE':<20} | {'-':<8} | {'-':<8} | {class_avg['Acc@1']*100:6.2f}%  | {class_avg['Acc@5']*100:6.2f}%  | {class_avg['T2I_R@1']*100:6.2f}%  | {class_avg['I2T_R@1']*100:6.2f}%")
    print("=" * 115)

if cap_results:
    print("\n" + "=" * 65)
    print("IMAGE-CAPTION BENCHMARKS EVALUATION RESULTS (Git-RSCLIP)")
    print("=" * 65)
    print(f"{'Dataset':<25} | {'Images':<8} | {'T2I R@1':<8} | {'I2T R@1':<8}")
    print("-" * 65)
    cap_avg = defaultdict(float)
    for ds_name, r in sorted(cap_results.items()):
        print(f"{ds_name:<25} | {r['num_images']:<8,} | {r['T2I_R@1']*100:6.2f}%  | {r['I2T_R@1']*100:6.2f}%")
        for k in ["T2I_R@1", "I2T_R@1"]: cap_avg[k] += r[k]
    print("-" * 65)
    for k in cap_avg: cap_avg[k] /= len(cap_results)
    print(f"{'AVERAGE':<25} | {'-':<8} | {cap_avg['T2I_R@1']*100:6.2f}%  | {cap_avg['I2T_R@1']*100:6.2f}%")
    print("=" * 65)

/data/susanket/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Using device: cuda
Loading Git-RSCLIP model and processor: lcybuaa/Git-RSCLIP


Loading weights: 100%|██████████| 792/792 [00:00<00:00, 3906.91it/s]


Model and processor successfully loaded!

[Image-Class] Evaluating CLRS... 

### FarSLIP

In [4]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', "3")

import io
import time
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import open_clip
from huggingface_hub import hf_hub_download

# ==========================================
# Configuration
# ==========================================
BENCHMARK_ROOT = "/data/susanket/vlm/eval"
MODEL_ARCH = "ViT-B-16"  # Architecture variant corresponding to FarSLIP2_ViT-B-16.pt
BATCH_SIZE = 64
IMAGE_SIZE = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ==========================================
# 1. Load FarSLIP Model & Tokenizer Manually
# ==========================================
print("Downloading FarSLIP checkpoint from Hugging Face...")
CKPT_PATH = hf_hub_download(repo_id="ZhenShiL/FarSLIP", filename="FarSLIP2_ViT-B-16.pt")
print(f"Checkpoint downloaded at: {CKPT_PATH}")

print(f"Initializing OpenCLIP architecture: {MODEL_ARCH}")
# Force context length to 248 to match FarSLIP's positional embedding shape
model, _, image_transform = open_clip.create_model_and_transforms(
    MODEL_ARCH, pretrained=None, force_quick_gelu=True, force_context_length=248
)

print(f"Loading custom state dict from: {CKPT_PATH}")
checkpoint = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

if "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
elif "model" in checkpoint:
    state_dict = checkpoint["model"]
else:
    state_dict = checkpoint

# Clean up DDP prefix if present
state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

# Load weights with strict=False
msg = model.load_state_dict(state_dict, strict=False)
print(f"State dict loaded successfully! Missing/Unexpected keys ignored: {msg}")

model = model.to(device)
model.eval()

tokenizer = open_clip.get_tokenizer(MODEL_ARCH)
print("FarSLIP model and tokenizer successfully loaded!")


# ==========================================
# 2. Datasets
# ==========================================
class ImageFolderFlatDataset(Dataset):
    """Loads images from class subfolders (raw PIL Images for transform handling)."""
    def __init__(self, root: str, image_size: int = 224):
        self.root = Path(root)
        self.image_size = image_size
        self.samples: List[Tuple[Path, str]] = []
        self.class_names: List[str] = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        self.class_to_idx = {name: i for i, name in enumerate(self.class_names)}

        for cls_name in self.class_names:
            cls_dir = self.root / cls_name
            for img_path in sorted(cls_dir.iterdir()):
                if img_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}:
                    self.samples.append((img_path, cls_name))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, cls_name = self.samples[idx]
        try:
            image = Image.open(path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (self.image_size, self.image_size))
        return image, self.class_to_idx[cls_name]


class ParquetCaptionDataset(Dataset):
    """Loads image-caption pairs from a Parquet file."""
    def __init__(self, parquet_path: str, image_size: int = 224):
        self.df = pd.read_parquet(parquet_path)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        caption = str(row["caption"])
        img_bytes = row["image"]

        try:
            image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        except Exception:
            image = Image.new("RGB", (self.image_size, self.image_size))

        return image, caption


# ==========================================
# 3. Feature Extraction & Metrics Functions
# ==========================================
def encode_images_in_batches(model, image_transform, dataloader: DataLoader, device: torch.device):
    model.eval()
    all_embeddings, all_targets = [], []
    
    with torch.no_grad():
        for images, targets in dataloader:
            # Apply OpenCLIP image transform pipeline on batch items
            pil_images = list(images) if not isinstance(images, list) else images
            tensor_batch = torch.stack([image_transform(img) for img in pil_images]).to(device)
            
            embeddings = model.encode_image(tensor_batch)
            embeddings = F.normalize(embeddings, dim=-1)
            all_embeddings.append(embeddings.cpu().numpy())
            
            if isinstance(targets, torch.Tensor):
                all_targets.extend(targets.cpu().numpy().tolist())
            elif isinstance(targets, (list, np.ndarray)):
                all_targets.extend(list(targets))
            else:
                all_targets.append(int(targets))

    img_embeddings = np.concatenate(all_embeddings, axis=0)
    img_labels = np.array(all_targets, dtype=int)

    return img_embeddings, img_labels


def encode_prompts(model, tokenizer, prompts: List[str], device: torch.device, batch_size: int = 128):
    model.eval()
    all_embeddings = []
    
    # Get context length dynamically from the model instance
    ctx_len = getattr(model, "context_length", 248)
    
    with torch.no_grad():
        for i in range(0, len(prompts), batch_size):
            batch_prompts = prompts[i:i + batch_size]
            # Explicitly pass context_length so the tokenizer outputs length 248 instead of 77
            tokens = tokenizer(batch_prompts, context_length=ctx_len).to(device)
            embeddings = model.encode_text(tokens)
            embeddings = F.normalize(embeddings, dim=-1)
            all_embeddings.append(embeddings.cpu().numpy())
            
    return np.concatenate(all_embeddings, axis=0)


def compute_retrieval_metrics(sim_matrix: np.ndarray, is_1to1: bool = False, img_labels: np.ndarray = None):
    n_text, n_img = sim_matrix.shape

    t2i_r1, t2i_r5, t2i_r10 = 0.0, 0.0, 0.0
    for i in range(n_text):
        ranks = np.argsort(-sim_matrix[i, :])
        matched = (ranks == i) if is_1to1 else (img_labels[ranks] == i)
        hit_ranks = np.where(matched)[0]
        if len(hit_ranks) > 0:
            if hit_ranks[0] < 1: t2i_r1 += 1.0
            if hit_ranks[0] < 5: t2i_r5 += 1.0
            if hit_ranks[0] < 10: t2i_r10 += 1.0

    t2i_res = {"T2I_R@1": t2i_r1 / n_text, "T2I_R@5": t2i_r5 / n_text, "T2I_R@10": t2i_r10 / n_text}

    i2t_r1, i2t_r5, i2t_r10 = 0.0, 0.0, 0.0
    for j in range(n_img):
        ranks = np.argsort(-sim_matrix[:, j])
        matched = (ranks == j) if is_1to1 else (ranks == img_labels[j])
        hit_ranks = np.where(matched)[0]
        if len(hit_ranks) > 0:
            if hit_ranks[0] < 1: i2t_r1 += 1.0
            if hit_ranks[0] < 5: i2t_r5 += 1.0
            if hit_ranks[0] < 10: i2t_r10 += 1.0

    i2t_res = {"I2T_R@1": i2t_r1 / n_img, "I2T_R@5": i2t_r5 / n_img, "I2T_R@10": i2t_r10 / n_img}
    return {**t2i_res, **i2t_res}


def compute_classification_accuracy(sim_matrix: np.ndarray, img_labels: np.ndarray):
    logits = sim_matrix.T  
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    top1_correct, top5_correct, n_samples = 0, 0, len(img_labels)

    for i in range(n_samples):
        ranking = np.argsort(-probs[i])
        if img_labels[i] == ranking[0]:
            top1_correct += 1
        if img_labels[i] in ranking[:5]:
            top5_correct += 1

    return {
        "Acc@1": top1_correct / n_samples,
        "Acc@5": top5_correct / min(5, probs.shape[1]) if probs.shape[1] >= 5 else top1_correct / n_samples
    }


# ==========================================
# 4. Run Evaluation Suite
# ==========================================
benchmark_root = Path(BENCHMARK_ROOT)
class_dir = benchmark_root / "image-class"
cap_dir = benchmark_root / "image-cap"

class_results = {}
cap_results = {}

def custom_collate(batch):
    images = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    if isinstance(targets[0], str):
        return images, targets
    return images, np.array(targets)

if class_dir.exists():
    for ds in sorted([d for d in class_dir.iterdir() if d.is_dir()]):
        print(f"\n[Image-Class] Evaluating {ds.name}...", end=" ", flush=True)
        t0 = time.time()
        dataset = ImageFolderFlatDataset(str(ds), image_size=IMAGE_SIZE)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=custom_collate)
        
        img_emb, img_labels = encode_images_in_batches(model, image_transform, loader, device)
        prompts = [f"{cls.replace('_', ' ')}" for cls in dataset.class_names]
        txt_emb = encode_prompts(model, tokenizer, prompts, device)
        
        sim_matrix = txt_emb @ img_emb.T
        retrieval_metrics = compute_retrieval_metrics(sim_matrix, is_1to1=False, img_labels=img_labels)
        cls_metrics = compute_classification_accuracy(sim_matrix, img_labels)
        
        class_results[ds.name] = {"num_images": len(dataset), "num_classes": len(dataset.class_names), **retrieval_metrics, **cls_metrics}
        print(f"done ({time.time()-t0:.1f}s) | Acc@1: {class_results[ds.name]['Acc@1']*100:.2f}%")

if cap_dir.exists():
    for p_file in sorted(list(cap_dir.glob("*.parquet"))):
        ds_name = p_file.stem
        print(f"\n[Image-Cap] Evaluating {ds_name}...", end=" ", flush=True)
        t0 = time.time()
        dataset = ParquetCaptionDataset(str(p_file), image_size=IMAGE_SIZE)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=custom_collate)
        
        img_emb, captions = encode_images_in_batches(model, image_transform, loader, device)
        txt_emb = encode_prompts(model, tokenizer, captions, device)
        
        sim_matrix = txt_emb @ img_emb.T
        retrieval_metrics = compute_retrieval_metrics(sim_matrix, is_1to1=True)
        
        cap_results[ds_name] = {"num_images": len(dataset), "T2I_R@1": retrieval_metrics["T2I_R@1"], "I2T_R@1": retrieval_metrics["I2T_R@1"]}
        print(f"done ({time.time()-t0:.1f}s) | T2I R@1: {cap_results[ds_name]['T2I_R@1']*100:.2f}%")


# ==========================================
# 5. Print Output Tables
# ==========================================
if class_results:
    print("\n" + "=" * 115)
    print("IMAGE-CLASS BENCHMARKS EVALUATION RESULTS (FarSLIP)")
    print("=" * 115)
    print(f"{'Dataset':<20} | {'Images':<8} | {'Classes':<8} | {'Acc@1':<8} | {'Acc@5':<8} | {'T2I R@1':<8} | {'I2T R@1':<8}")
    print("-" * 115)
    class_avg = defaultdict(float)
    for ds_name, r in sorted(class_results.items()):
        print(f"{ds_name:<20} | {r['num_images']:<8,} | {r['num_classes']:<8} | {r['Acc@1']*100:6.2f}%  | {r['Acc@5']*100:6.2f}%  | {r['T2I_R@1']*100:6.2f}%  | {r['I2T_R@1']*100:6.2f}%")
        for k in ["Acc@1", "Acc@5", "T2I_R@1", "I2T_R@1"]: class_avg[k] += r[k]
    print("-" * 115)
    for k in class_avg: class_avg[k] /= len(class_results)
    print(f"{'AVERAGE':<20} | {'-':<8} | {'-':<8} | {class_avg['Acc@1']*100:6.2f}%  | {class_avg['Acc@5']*100:6.2f}%  | {class_avg['T2I_R@1']*100:6.2f}%  | {class_avg['I2T_R@1']*100:6.2f}%")
    print("=" * 115)

if cap_results:
    print("\n" + "=" * 65)
    print("IMAGE-CAPTION BENCHMARKS EVALUATION RESULTS (FarSLIP)")
    print("=" * 65)
    print(f"{'Dataset':<25} | {'Images':<8} | {'T2I R@1':<8} | {'I2T R@1':<8}")
    print("-" * 65)
    cap_avg = defaultdict(float)
    for ds_name, r in sorted(cap_results.items()):
        print(f"{ds_name:<25} | {r['num_images']:<8,} | {r['T2I_R@1']*100:6.2f}%  | {r['I2T_R@1']*100:6.2f}%")
        for k in ["T2I_R@1", "I2T_R@1"]: cap_avg[k] += r[k]
    print("-" * 65)
    for k in cap_avg: cap_avg[k] /= len(cap_results)
    print(f"{'AVERAGE':<25} | {'-':<8} | {cap_avg['T2I_R@1']*100:6.2f}%  | {cap_avg['I2T_R@1']*100:6.2f}%")
    print("=" * 65)

Using device: cuda
Checkpoint downloaded at: /home/susanket/.cache/huggingface/hub/models--ZhenShiL--FarSLIP/snapshots/9742933cfc3e7e62f6674cdb15ead7d1a9d148f3/FarSLIP2_ViT-B-16.pt
Initializing OpenCLIP architecture: ViT-B-16


Loading custom state dict from: /home/susanket/.cache/huggingface/hub/models--ZhenShiL--FarSLIP/snapshots/9742933cfc3e7e62f6674cdb15ead7d1a9d148f3/FarSLIP2_ViT-B-16.pt
State dict loaded successfully! Missing/Unexpected keys ignored: _IncompatibleKeys(missing_keys=[], unexpected_keys=['positional_embedding_res', 'mask1', 'mask2'])
FarSLIP model and tokenizer successfully loaded!

[Image-Class] Evaluating CLRS... done (51.1s) | Acc@1: 66.97%

[Image-Class] Evaluating NWPU_images... done (19.3s) | Acc@1: 84.78%

[Image-Class] Evaluating Optimal31... done (7.2s) | Acc@1: 79.52%

[Image-Class] Evaluating RSC11... done (7.3s) | Acc@1: 76.84%

[Image-Class] Evaluating RSICB128... done (102.2s) | Acc@1: 42.88%

[Image-Class] Evaluating RSSCN7... done (14.5s) | Acc@1: 74.07%

[Image-Class] Evaluating WHU-RS19... done (8.7s) | Acc@1: 84.78%

[Image-Cap] Evaluating RSICD... 

ValueError: invalid literal for int() with base 10: 'The airport is very large.next to the airport is the green grass.next to the airport is green grass .The airport is very large.the airport is very large .'